# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!git clone https://github.com/ainasarfaraz343-a11y/flyrank-internship.git
%cd flyrank-internship

Cloning into 'flyrank-internship'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 138 (delta 45), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 1.87 MiB | 5.16 MiB/s, done.
Resolving deltas: 100% (45/45), done.
/content/flyrank-internship


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

np.random.seed(42)

In [ ]:
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

feature_cols = ['days_since_last_update', 'impressions_90d', 'clicks_90d',
                 'ctr', 'avg_position', 'engagement_rate', 'word_count',
                 'search_volume', 'content_age_days']

model_df = df[df['avg_position'] > 0].copy()
for col in ['word_count', 'search_volume', 'engagement_rate']:
    model_df[f'has_{col}'] = model_df[col].notna().astype(int)
    model_df[col] = model_df[col].fillna(0)
feature_cols = feature_cols + ['has_word_count', 'has_search_volume', 'has_engagement_rate']

X = model_df[feature_cols]
y = model_df['is_declining']
groups = model_df['client_id']

print(f"Loaded {len(df)} rows, {len(model_df)} after filtering.")

Loaded 30000 rows, 28795 after filtering.


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1: The Health Score formula**

The paper builds its composite "Health Score" out of four components:
Impressions (30 points), Position (30 points), CTR (20 points), and Scroll
Depth (20 points).

What I keep coming back to is where the 30/30/20/20 split came from. Was it
tuned against some outcome — pages that later grew vs. declined, say — or is
it a reasonable-sounding judgment call? The paper doesn't say, and it matters
here more than it would for a simple average, because two of the four inputs
aren't independent. A page's position drives its impressions almost by
definition — rank higher, get seen more. Giving both a full 30 points each
risks weighting one real signal (visibility) twice under two different names,
while scroll depth and CTR — which arguably capture something more distinct —
get comparatively less say in the final number. None of this means the score
is wrong, just that I'd want to see it validated against an actual outcome
before treating "22.3" as a meaningful cutoff rather than a rough sort order.

**Finding 2: What "80/20 split" actually means**

The ML pipeline section mentions Random Forest and Logistic Regression, each
trained on an 80/20 split across ~61.8K content pieces from 57 brands — but
it doesn't say whether that split was random or grouped by brand.

This isn't a minor detail. In my own Week-6 audit, running the same model
under a random split versus a client-grouped split produced a measurable
gap in ROC-AUC — the random split let the model partially memorize
client-specific patterns rather than learn anything that generalizes. If
this paper's 80/20 split is a random row split across 57 brands, its ML
numbers could be inflated the same way mine were before I switched to a
grouped split. To be fair to the paper, it explicitly says the ML section is
"exploratory appendix material" and that the headline findings lean on direct
aggregate comparisons instead — so this question doesn't threaten the paper's
main conclusions. It's still worth asking, because it's the exact failure
mode this course has spent two weeks teaching me to check for in my own work.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

I already used a grouped split back in Week 5, but I wanted to actually see
the difference it makes, not just assume it matters. So I trained the exact
same Logistic Regression twice — once with a normal random 80/20 split, and
once with the split grouped by client_id like before.

The random split came out to 0.623 ROC-AUC, and the grouped one came out to
0.559. That's about a 6-point gap, and honestly it makes sense once you think
about it — when rows from the same client can land in both train and test,
the model gets a bit of a "preview" of that client's behavior before it's
even tested on them. So the 0.623 wasn't really earned, it was partly just
the model recognizing a client it had already seen bits of. The 0.559 number
is the one I'm treating as real from here on, since it's actually asking
whether this works on a client the model has never touched.

In [ ]:
# BEFORE: random split (dishonest)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler_r = StandardScaler()
X_train_r_s = scaler_r.fit_transform(X_train_r)
X_test_r_s = scaler_r.transform(X_test_r)
clf_r = LogisticRegression(max_iter=1000, random_state=42)
clf_r.fit(X_train_r_s, y_train_r)
random_auc = roc_auc_score(y_test_r, clf_r.predict_proba(X_test_r_s)[:, 1])

# AFTER: grouped split (honest)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

scaler_g = StandardScaler()
X_train_s = scaler_g.fit_transform(X_train)
X_test_s = scaler_g.transform(X_test)
clf_g = LogisticRegression(max_iter=1000, random_state=42)
clf_g.fit(X_train_s, y_train)
model_scores = clf_g.predict_proba(X_test_s)[:, 1]
grouped_auc = roc_auc_score(y_test, model_scores)

before_after = pd.DataFrame({
    'split_type': ['Random split (dishonest)', 'Grouped by client_id (honest)'],
    'ROC-AUC': [round(random_auc, 3), round(grouped_auc, 3)]
})
print(before_after)
print(f"Gap: {round(random_auc - grouped_auc, 3)}")

                      split_type  ROC-AUC
0       Random split (dishonest)    0.623
1  Grouped by client_id (honest)    0.559
Gap: 0.064


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

For this I mostly wanted to check my own feature list wasn't hiding anything
it shouldn't. First I just listed out `trend_direction` and `trend_pct` since
those are literally what the label comes from, and confirmed neither one is
in my features — they're not, so that part's clean.

Then I did the test the skill recommends, which is to break my own model on
purpose and see if it actually breaks. I added `trend_pct` in as a feature
even though it shouldn't be there, just to see what happens. AUC went from
0.559 to a flat 1.000. That's basically the model being handed the answer key
— if the score jumps to near-perfect like that, it's a pretty reliable sign
something's leaking. Since my real model doesn't include that column, this
was really just a check that I'd actually notice if it did. Split overlap
between train and test clients came back empty too, and I never used the
Week-4 rule's own output as an input anywhere, only as the thing I'm
comparing against.

In [ ]:
print("LEAKAGE AUDIT CHECKLIST")
print("="*50)

suspect_cols = ['trend_direction', 'trend_pct']
in_features = [c for c in suspect_cols if c in feature_cols]
print(f"1. Label-derived columns in features: {in_features if in_features else 'NONE — clean'}")

X_leaky = X_train.copy()
X_leaky['trend_pct_LEAKY'] = model_df.iloc[train_idx]['trend_pct'].fillna(0).values
X_test_leaky = X_test.copy()
X_test_leaky['trend_pct_LEAKY'] = model_df.iloc[test_idx]['trend_pct'].fillna(0).values

scaler_l = StandardScaler()
X_leaky_s = scaler_l.fit_transform(X_leaky)
X_test_leaky_s = scaler_l.transform(X_test_leaky)
clf_leaky = LogisticRegression(max_iter=1000, random_state=42)
clf_leaky.fit(X_leaky_s, y_train)
leaky_auc = roc_auc_score(y_test, clf_leaky.predict_proba(X_test_leaky_s)[:, 1])

print(f"\n2. Sanity check — deliberately adding trend_pct:")
print(f"   Honest model AUC: {grouped_auc:.3f}  |  With leaky feature AUC: {leaky_auc:.3f}")

print(f"\n3. Product/decision-derived flags (Week-4 rule score) as features: NONE")
print(f"   (used only as the baseline to beat, never as model input)")

overlap = set(model_df.iloc[train_idx]['client_id']) & set(model_df.iloc[test_idx]['client_id'])
print(f"\n4. Split grouped by client_id — overlap: {overlap if overlap else 'zero, confirmed'}")

print(f"\n5. Base rate printed next to metric: {y_test.mean():.3f}")

LEAKAGE AUDIT CHECKLIST
1. Label-derived columns in features: NONE — clean

2. Sanity check — deliberately adding trend_pct:
   Honest model AUC: 0.559  |  With leaky feature AUC: 1.000

3. Product/decision-derived flags (Week-4 rule score) as features: NONE
   (used only as the baseline to beat, never as model input)

4. Split grouped by client_id — overlap: zero, confirmed

5. Base rate printed next to metric: 0.540


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Looking back at Week 5, the line I'd flag as too confident is where I said
Logistic Regression "outperforms" the Week-4 rule at precision@50. That word
outperforms makes it sound more settled than it actually is — like it's a
proven fact rather than something I measured once, on one split.

What I can actually back up is this: on the measured test split, Logistic
Regression showed an observed, directional edge over the Week-4 rule at
precision@50 (0.44 vs 0.28). It's decision-support for figuring out what to
prioritize first, not a guarantee that it'll hold up the same way on a
different set of clients or further down the line — I only have one split's
worth of evidence, and the Section 2 comparison already showed how much a
split choice alone can shift a number.

It's also worth saying plainly that the model's overall discrimination is
weak — ROC-AUC sits around 0.56 even under the honest grouped split, which
is close to a coin flip. So this is a small, observed signal that's useful
near the top of a ranked list, not evidence of a strong general classifier,
and I don't want to present it as one.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.